In [ ]:
%reload_ext autoreload
%autoreload 2

In [ ]:
# Dependencies - General Stuff
import numpy as np
import importlib
import tempfile
from scipy.signal import find_peaks

# strfpy
from strfpy import preprocSound, strfSetup, trnDirectFit, calcSegmentedModel
from strfpy.timeFreq import timefreq_raw
from soundsig.sound import spec_colormap

import pynwb
from matplotlib import pyplot as plt
# %matplotlib widget
plt.ion()

#### Read the data file

In [ ]:
# nwb_file = '/aquila_ssd2/lthomas/songephys_data/OperantEphys/HpiPur2667F/sites/HpiPur2667F_site02_240905_072851_pb_op/HpiPur2667F_site02_240905_072851_pb_op_ks4_lat_250215.nwb'

# nwb_file = '/Users/frederictheunissen/Working Data/OperantEphys/NWB_Files/HpiPur2667F_site02_240905_072851_pb_op_ks4_lat_250215.nwb'
nwb_file = '/Users/kailinzhuang/research-desktop/data/OperantEphys/HpiPur2667F/HpiPur2667F_site02_240905_072851_pb_op_ks4_lat_250215.nwb'
# Load the nwb file
preprocOptions = {} # we'll leave this empty and use default options
nwb_io =  pynwb.NWBHDF5IO(nwb_file, mode='r')
nwb = nwb_io.read()
units = nwb.units.to_dataframe()
# load the good units
good_units = units[units.group == 'good']


In [ ]:
# sample a random unit
unit = good_units.sample().iloc[0]
#unit = good_units.iloc[10]       # Unit iloc[10] good example for a neuron with strong onset-offset response - positive onset and negative offset
unit = good_units.iloc[7]
print("Processing unit: ", unit.name)

# These are units to check 49, 64, 83 - in good units they are 23, xx, xx

In [ ]:
nwb

In [ ]:
unit = good_units.sample()
unit

In [ ]:
unit.iloc[0]

In [ ]:
# sample a random unit
unit = good_units.sample().iloc[0]
#unit = good_units.iloc[10]       # Unit iloc[10] good example for a neuron with strong onset-offset response - positive onset and negative offset
unit = good_units.iloc[7]
print("Processing unit: ", unit.name)

# These are units to check 49, 64, 83 - in good units they are 23, xx, xx

### Plot the stim and microphone data of a random playback trial

In [ ]:
def get_mic_data(nwb, trial):
    rate = nwb.acquisition['audio'].rate
    mic_data = nwb.acquisition['audio'].data
    start_id = int(trial.start_time * rate)
    end_id = int(trial.stop_time * rate)
    mic_trial = mic_data[start_id:end_id]
    return mic_trial[:,1], mic_trial[:,0]

In [ ]:
# lets take one trial and compare the spectrogram to the spectrogram of the microphone data
# lets get a ranodm trial
trials = nwb.intervals['playback_trials'].to_dataframe()
print(trials.columns)
print(trials['experiment_uid'].unique())

trials.tail()

Repeat the cell below to get another example

In [ ]:
trials.sample().iloc[0]

In [ ]:
# Choose a ramdom trial
trial = trials.sample().iloc[0]
mic_trial, mic_copy = get_mic_data(nwb, trial)
rate = nwb.acquisition['audio'].rate

# Spectrogram paramteters.
stim_params = {}
stim_params['fband'] = 120
stim_params['nstd'] = 6
stim_params['high_freq'] = 8000
stim_params['low_freq'] = 250
stim_params['log'] = 1
stim_params['stim_rate'] = 1000  # Sample rate of spectrogram
DBNOISE = 80
# Colormap for plotting spectrograms
spec_colormap()   # defined in sound.py


# First figure for the microphone copy
f, (ax1, ax2) = plt.subplots(2, 1, sharex=True, dpi=100, figsize = (8,4))

tfrep = timefreq_raw(mic_copy, rate, 'ft', params=stim_params)
cmap = plt.get_cmap('SpectroColorMap')

minSpect = tfrep['spec'].max()-DBNOISE
maxB = tfrep['spec'].max()
ax1.imshow(tfrep['spec'], extent=[tfrep['t'][0], tfrep['t'][-1], tfrep['f'][0]*1e-3, tfrep['f'][-1]*1e-3],
                aspect='auto', interpolation='nearest', origin='lower', cmap=cmap, vmin=minSpect, vmax=maxB)
ax1.set_ylim(0, 8)
ax1.set_ylabel('Frequency (kHz)')

tval = np.arange(stop=len(mic_copy))/rate
ax2.plot(tval, mic_copy)
ax2.set_xlabel('Time (s)')

# Second copy for the microphone 
f, (ax1, ax2) = plt.subplots(2, 1, sharex=True, dpi=100, figsize = (8,4))

tfrep = timefreq_raw(mic_trial, rate, 'ft', params=stim_params)
minSpect = tfrep['spec'].max()-DBNOISE
maxB = tfrep['spec'].max()
ax1.imshow(tfrep['spec'], extent=[tfrep['t'][0], tfrep['t'][-1], tfrep['f'][0]*1e-3, tfrep['f'][-1]*1e-3],
                aspect='auto', interpolation='nearest', origin='lower', cmap=cmap, vmin=minSpect, vmax=maxB)
ax1.set_ylim(0, 8)
ax1.set_ylabel('Frequency (kHz)')

tval = np.arange(stop=len(mic_trial))/rate
ax2.plot(tval, mic_trial)
ax2.set_xlabel('Time (s)')



#### Create an stimulus-response data set (srData) per stimlus

In [ ]:
# Preprocess sound stimulus and segment

respChunkLen = 100 # ms of stim to use in each chunk of feature space
segmentBuffer = 30 # ms to add at the beginning of each segment
nLaguerre = 25 # number of laguerre functions to use
feature = 'spect_windows'
event_types = 'onoff_feature'
nPoints = 150 # number of points to use in the kernel

srData = preprocSound.preprocess_sound_nwb(nwb_file, 'playback_trials', unit.name, preprocess_type='ft')
calcSegmentedModel.preprocess_srData(srData, plot=False, respChunkLen=respChunkLen, segmentBuffer=segmentBuffer, tdelta=0, plotFlg = False)

print('This playback stim-response data set has %d stimuli.' % (len(srData['datasets'])))


In [ ]:
srData

In [ ]:
# Estimate the single trial SNR for this data set - should we use the same mean?
smWindow = 31  # smoothing window in ms
snrEst, f, snrEstf, cumInfo, totWeight = preprocSound.estimate_SNR(srData, smWindow=smWindow)
print('The single trial SNR with smoothing at %.0f ms is %.4f' % (smWindow, 
                                                                  snrEst))
print('The corresponding EV for 1 trials is %.4f' % (snrEst/(snrEst + 1)))
print('The corresponding EV for 10 trials is %.4f' % (10*snrEst/(10*snrEst + 1)))
print('The cumulative Information up to %.0f Hz is %.4f bits/s' % (f[-1], cumInfo[-1]))
print('The total weight (data points used) is %.0f' % (totWeight))

# Plot the SNRf and the Cumulative Information
figname = 'SNR and Information for unit %s' % (unit.name)
fig, ax1 = plt.subplots(1,1, dpi=100, figsize=(6,6))
fig.suptitle(figname)
ax1.plot(f, snrEstf/(snrEstf+1), 'k')
ax1.set_ylabel('Coherence')
# ax1.set_yscale('log')
ax2 = ax1.twinx()
ax2.plot(f, cumInfo, 'r')
ax2.set_ylabel('Cumulative Info(f) [bits/s/Hz]')
ax2.set_xlabel('Frequency (Hz)')    
ax2.set_xlim([0, f[-1]/2])
